## 10. Modelo XGBoost — Predicción de Dengue Grave

Entrena un clasificador binario (¿hubo algún caso de dengue grave en el mes?) usando 30 features construidas en el notebook 09. Los experimentos se registran en MLflow.

**Variable objetivo:** `grave > 0` (mes con al menos un caso grave)  
**Granularidad:** mensual por municipio (DIVIPOLA)  
**Partición temporal:** train 2007-2021 / val 2022 / test 2023-2024

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)
import warnings, os
warnings.filterwarnings("ignore")

# ── Configuracion MLflow ────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME     = "dengue-grave-xgboost"

DATA_PATH = "../data/processed/dengue_features_modelado.csv"
TARGET    = "grave"
FEATURE_COLS = (
    [f"grave_lag_{l}"   for l in [1,2,3,4,6]] +
    [f"clasico_lag_{l}" for l in [1,2,3,4,6]] +
    ["grave_roll3", "clasico_roll3"] +
    ["temp_mean_c", "temp_lag_1", "temp_lag_2", "temp_lag_3"] +
    ["rain_mm_day", "rain_lag_1", "rain_lag_2", "rain_lag_3"] +
    ["mes_sin", "mes_cos", "anio_epidemia", "ANO", "MES"] +
    ["es_endemico", "zona_canal_lag1", "p25", "p75", "sir_lag1"]
)  # 30 features mensuales

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

### 1. Carga de datos

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"ANO range: {df['ANO'].min()} - {df['ANO'].max()}")
print(f"Columnas disponibles: {list(df.columns)}")

### 2. Preparacion del target

La variable `grave` es un conteo mensual de casos. La convertimos a binaria: **1** si hubo al menos un caso, **0** si no.

In [ ]:
df["grave_bin"] = (df[TARGET] > 0).astype(int)

pos = df["grave_bin"].mean()
print(f"Prevalencia global: {pos*100:.1f}% positivos ({df['grave_bin'].sum():,} / {len(df):,} filas)")

### 3. Particion temporal

Se respeta el orden temporal para evitar data leakage. El modelo nunca ve datos del futuro durante el entrenamiento.

In [ ]:
train = df[df["ANO"] <= 2021].copy()
val   = df[df["ANO"] == 2022].copy()
test  = df[df["ANO"] >= 2023].copy()

for name, split in [("train", train), ("val", val), ("test", test)]:
    n_pos = split["grave_bin"].mean()
    print(f"{name:5s}: {len(split):>7,} filas | ANO {split['ANO'].min()}-{split['ANO'].max()} | {n_pos*100:.1f}% positivos")

X_train = train[FEATURE_COLS]; y_train = train["grave_bin"]
X_val   = val[FEATURE_COLS];   y_val   = val["grave_bin"]
X_test  = test[FEATURE_COLS];  y_test  = test["grave_bin"]

In [ ]:
neg_train = (y_train == 0).sum()
pos_train = (y_train == 1).sum()
spw = neg_train / pos_train
print(f"Negativos: {neg_train:,}  |  Positivos: {pos_train:,}")
print(f"scale_pos_weight recomendado: {spw:.1f}")

### 4. Funciones de evaluacion

In [ ]:
def evaluar(nombre, y_true, y_prob, threshold=0.5, log_mlflow=False):
    y_pred = (y_prob >= threshold).astype(int)
    metricas = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "auroc":     roc_auc_score(y_true, y_prob),
        "avg_precision": average_precision_score(y_true, y_prob),
    }
    print(f"\n=== {nombre} ===")
    for k, v in metricas.items():
        print(f"  {k:15s}: {v:.4f}")
    if log_mlflow:
        mlflow.log_metrics({f"{nombre}_{k}": v for k, v in metricas.items()})
    return metricas

def matriz_confusion(nombre, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nMatriz de confusion ({nombre}):")
    print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")
    return cm

### 5. Modelo baseline

Un clasificador que predice la clase mas frecuente (cero). Es el piso que cualquier modelo real debe superar.

In [ ]:
with mlflow.start_run(run_name="baseline-stratified"):
    dummy = DummyClassifier(strategy="stratified", random_state=42)
    dummy.fit(X_train, y_train)

    prob_val_d  = dummy.predict_proba(X_val)[:, 1]
    prob_test_d = dummy.predict_proba(X_test)[:, 1]

    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "stratified")

    metricas_val_d  = evaluar("val",  y_val,  prob_val_d,  log_mlflow=True)
    metricas_test_d = evaluar("test", y_test, prob_test_d, log_mlflow=True)

print("\nBaseline completado.")

### 6. XGBoost con manejo de desbalance

`scale_pos_weight` compensa el desbalance 90/10: el modelo penaliza mas los falsos negativos (casos graves que no detecta).

In [ ]:
params_xgb = {
    "n_estimators":     500,
    "max_depth":        6,
    "learning_rate":    0.05,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": spw,
    "eval_metric":      "aucpr",
    "early_stopping_rounds": 30,
    "random_state":     42,
    "n_jobs":           -1,
    "tree_method":      "hist",
}

with mlflow.start_run(run_name="xgboost-v1") as run:
    mlflow.log_params({
        k: v for k, v in params_xgb.items()
        if k not in ["early_stopping_rounds", "random_state", "n_jobs", "tree_method"]
    })
    mlflow.log_param("features", len(FEATURE_COLS))
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("train_pos_pct", round(y_train.mean()*100, 2))

    model_xgb = xgb.XGBClassifier(**params_xgb, use_label_encoder=False)
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )

    best_iter = model_xgb.best_iteration
    mlflow.log_param("best_iteration", best_iter)
    print(f"\nMejor iteracion: {best_iter}")

    prob_val_xgb  = model_xgb.predict_proba(X_val)[:, 1]
    prob_test_xgb = model_xgb.predict_proba(X_test)[:, 1]

    metricas_val  = evaluar("val",  y_val,  prob_val_xgb,  log_mlflow=True)
    metricas_test = evaluar("test", y_test, prob_test_xgb, log_mlflow=True)

    matriz_confusion("val",  y_val,  prob_val_xgb)
    matriz_confusion("test", y_test, prob_test_xgb)

    mlflow.xgboost.log_model(model_xgb, artifact_path="model")
    run_id = run.info.run_id
    print(f"\nrun_id: {run_id}")

### 7. Importancia de features

In [ ]:
imp = pd.Series(
    model_xgb.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
imp.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 20 features por importancia (gain)", fontsize=13)
ax.set_xlabel("Importancia")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../data/figures/12_feature_importance.png", dpi=120)
plt.close()

with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact("../data/figures/12_feature_importance.png")

print(imp.head(10).to_string())

### 8. Curvas ROC y Precision-Recall (validacion)

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, prob_val_xgb)
fpr_d,   tpr_d,   _ = roc_curve(y_val, prob_val_d)
axes[0].plot(fpr_xgb, tpr_xgb, color='steelblue',
             label=f'XGBoost (AUROC={auc(fpr_xgb, tpr_xgb):.3f})')
axes[0].plot(fpr_d, tpr_d, color='gray', linestyle='--',
             label=f'Baseline (AUROC={auc(fpr_d, tpr_d):.3f})')
axes[0].plot([0, 1], [0, 1], color='black', linestyle=':')
axes[0].set_xlabel('Tasa de falsos positivos')
axes[0].set_ylabel('Tasa de verdaderos positivos')
axes[0].set_title('Curva ROC - Validacion 2022')
axes[0].legend()

# Precision-Recall
prec_xgb, rec_xgb, _ = precision_recall_curve(y_val, prob_val_xgb)
prec_d,   rec_d,   _ = precision_recall_curve(y_val, prob_val_d)
axes[1].plot(rec_xgb, prec_xgb, color='steelblue',
             label=f'XGBoost (AP={average_precision_score(y_val, prob_val_xgb):.3f})')
axes[1].plot(rec_d, prec_d, color='gray', linestyle='--',
             label=f'Baseline (AP={average_precision_score(y_val, prob_val_d):.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall - Validacion 2022')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/figures/13_curvas_roc_pr.png', dpi=120)
plt.close()

with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact('../data/figures/13_curvas_roc_pr.png')

print('Figuras guardadas.')


### 9. Seleccion de umbral optimo (F1 en validacion)

El umbral por defecto (0.5) no es optimo con datos desbalanceados. Buscamos el umbral que maximiza el F1 en el conjunto de validacion.

In [ ]:
from sklearn.metrics import f1_score as f1

thresholds = np.arange(0.05, 0.95, 0.01)
f1s = [f1(y_val, (prob_val_xgb >= t).astype(int), zero_division=0) for t in thresholds]

best_thr = thresholds[np.argmax(f1s)]
print(f"Umbral optimo (max F1 val): {best_thr:.2f}")
print(f"F1 con umbral 0.50:         {f1(y_val, (prob_val_xgb>=0.50).astype(int), zero_division=0):.4f}")
print(f"F1 con umbral optimo:       {f1(y_val, (prob_val_xgb>=best_thr).astype(int), zero_division=0):.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1s, color="steelblue")
ax.axvline(best_thr, color="red", linestyle="--", label=f"Umbral optimo {best_thr:.2f}")
ax.set_xlabel("Umbral"); ax.set_ylabel("F1")
ax.set_title("F1 vs umbral de clasificacion (validacion 2022)")
ax.legend(); plt.tight_layout()
plt.savefig("../data/figures/14_umbral_optimo.png", dpi=120)
plt.close()

with mlflow.start_run(run_id=run_id):
    mlflow.log_param("best_threshold", round(float(best_thr), 2))
    mlflow.log_artifact("../data/figures/14_umbral_optimo.png")

### 10. Evaluacion final en conjunto de prueba (2023-2024)

Esta evaluacion se hace una sola vez, con el umbral optimo determinado en validacion.

In [ ]:
print("=== Evaluacion final — Test 2023-2024 ===")
print(f"Umbral aplicado: {best_thr:.2f}\n")

evaluar("test_optimo", y_test, prob_test_xgb, threshold=best_thr, log_mlflow=False)
matriz_confusion("test_optimo", y_test, prob_test_xgb, threshold=best_thr)

print("\n" + classification_report(
    y_test,
    (prob_test_xgb >= best_thr).astype(int),
    target_names=["no grave", "grave"],
    zero_division=0
))

with mlflow.start_run(run_id=run_id):
    metricas_test_opt = evaluar("test_opt", y_test, prob_test_xgb, threshold=best_thr)
    mlflow.log_metrics({f"test_opt_{k}": v for k, v in metricas_test_opt.items()})

print("\nModelo guardado en MLflow run_id:", run_id)

### 11. Resumen de resultados

| Modelo | Conjunto | AUROC | Avg Precision |
|---|---|---|---|
| Baseline (stratified) | val | ~0.50 | ~0.08 |
| **XGBoost v1 (mensual)** | **val** | **pendiente ejecucion** | **pendiente** |
| **XGBoost v1 (mensual)** | **test** | **pendiente ejecucion** | **pendiente** |

**Cambios respecto a version semanal:**
- Granularidad mensual (104,380 filas vs ~430k semanales): menos ruido, 58% zeros vs 82%
- DIVIPOLA 5 digitos como clave correcta de municipio
- Clima ERA5/CHIRPS via GEE (`temp_mean_c`, `rain_mm_day` + rezagos)
- `anio_epidemia = ANO - 2007` (tendencia continua, no binaria)
- Canal endemico: referencia 2007-2021 con fix P75=0
- Rezagos hasta 6 meses (no 8 semanas), rolling 3 meses

**Proximos pasos:**
- Ejecutar celda 14 para obtener metricas reales con datos mensuales
- Comparar con notebook 11 (GAM-Poisson, Regresion, Random Forest)
- Ajuste fino de hiperparametros segun resultados
